# TIDMAD: denoise one band, then test on unseen injected frequencies

A noisy detector waveform is the input; the injected clean waveform is the target. A model learns to recover that target. This tutorial uses **band `0-3`**: four training files and four validation files, not four separate model searches.

You will learn to (1) change search/data/time/VRAM settings, (2) exclude selected injected frequencies from training, and (3) reserve another frequency group for a final test after model selection. These are teaching experiments, not qualified reproductions of the paper scores.

Run this notebook from your **external project**, using the exp checkout's own Python kernel. Default Run All only explains and previews. It does not train, call APIs, scan raw data or modify files. Enable the named switches only when ready to save your own variants. Launch scripts separately in a terminal.

In [ ]:
import json, os, sys
from pathlib import Path

PROJECT = globals().get("PROJECT", Path.cwd().resolve())
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
binding = json.loads((PROJECT / "project.json").read_text())
EXP = Path(binding["exp_checkout"])
assert Path(sys.prefix).resolve() == (EXP / ".venv").resolve(), "Select this exp checkout's kernel"
os.chdir(EXP)  # Import from the exact checkout; generated files still go to PROJECT.
from tutorials.paper.tidmad.runner import TidmadExperiment, credential_status
from tutorials.paper.tidmad.project import write_launcher, write_frequency_task
from tasks.tidmad.runtime.frequency_split import FrequencyCatalog, FrequencyRow, make_split, sample_population

base = TidmadExperiment.model_validate_json(Path(binding["default_experiment"]).read_text())
print("Your project:", PROJECT)
print("Task:", base.composition)
print("Experiment:", binding["default_experiment"])
print("Output of a future launch:", base.workspace)

## 1. Find the setting you want to change

**Iteration** means another search iteration; **epoch** means a pass through that attempt's selected training data. **Trial** is the smaller feasibility run; **Formal** is the larger training/evaluation stage. Trials can fail or retry, so these are not exact counts of executions.

| Your question | Experiment JSON field | What it controls |
|---|---|---|
| Search longer? | `iterations` | Search iteration ceiling |
| Train more epochs? | `epochs` | Trial and Formal epoch ceilings |
| Use less Trial training data? | `trial_train_fraction` | `--trial_portion`; meaning depends on the task below |
| Evaluate less Trial data? | `trial_val_fraction` | `--eval_portion` |
| Use less Formal training data? | `formal_train_fraction` | `--formal_portion`; **locked to `.1` for paper-pool** |
| Evaluate less Formal data? | `formal_val_fraction` | `--formal_eval_portion` |
| Change time allowances? | `trial_minutes`, `formal_minutes` | Native per-stage budgets; not a total campaign deadline |
| Change memory allowances? | `trial_vram_gib`, `formal_vram_gib` | Native VRAM budgets in GiB; leave physical GPU headroom |

A smaller fraction is not a new split. It samples fewer members of the **same eligible population**. To exclude frequencies you must first change the task's population.

In [ ]:
def variant(settings, name, **changes):
    values = settings.model_dump(mode="json")
    values.update(changes, run_name=name, workspace=str(PROJECT / "runs" / name))
    return TidmadExperiment.model_validate(values)

longer = variant(base, "tidmad_longer_001", iterations=3, epochs=10)
less_trial = variant(base, "tidmad_small_trial_001", trial_train_fraction=.25, trial_val_fraction=.1)
budgets = variant(base, "tidmad_budget_001", trial_minutes=3, formal_minutes=10,
                  trial_vram_gib=6, formal_vram_gib=8)
for label, settings in [("longer", longer), ("less Trial", less_trial), ("budgets", budgets)]:
    print(label, {k: v for k, v in settings.model_dump(mode="json").items()
                  if v != base.model_dump(mode="json").get(k)})

## 2. Paper pool: understand the denominator before changing a proportion

Each file has 200 one-second PSD segments; each segment contains 10 million samples. The paper task fixes **20 training segments per file**. A model window remains 40,000 samples, so a full one-second segment contains 250 model windows.

For this frozen pool, Trial `.5` selects 10 of the 20 training segments per file. Formal `formal_train_fraction=.1` declares the original **20/200 parent pool**, and Formal uses all 20. It does **not** take another 10% of those 20. Changing that field to `.2` is refused by this teaching entrypoint.

Evaluation is independent: Trial `.1` samples from the 200 validation segments; Formal `1.0` evaluates all 200. This paper-pool example does **not** hold out frequencies. Tiny Trial scopes or short budgets can fail the existing calibration/Health checks; raise the allowance instead of disabling a check.

In [ ]:
print("Paper pool, per file:")
print("Trial training .50:", round(20 * .5), "/ 20 frozen segments")
print("Trial training .25:", round(20 * .25), "/ 20 frozen segments")
print("Formal training .10: all 20 frozen segments, originally 20 / 200")
print("Trial evaluation .10: drawn from 200 original validation segments")
print("Formal evaluation 1.0: all 200 original validation segments")

## 3. Frequency holdout: change eligibility in the task package

Suppose 1,100 Hz appears in several time segments in both source families. If that frequency belongs to **train**, we use its `abra_training_*` segments for training and discard its `abra_validation_*` segments from evaluation. A **validation** frequency uses only its `abra_validation_*` segments; its training-family segments are excluded. **Final-test** frequencies are excluded from both workflow training and workflow validation.

| Frequency group | Original training files | Original validation files | Who sees the score? |
|---|---|---|---|
| Train | Eligible for training | Excluded | No evaluation score |
| Validation | Excluded | Workflow evaluation | Agent; also used during training validation |
| Final test | Excluded | Separate inference after selection | Human, after search has stopped |

Here **mask** means exclude whole examples from the training population, not zero Fourier coefficients in a waveform. The same nominal injected frequency always stays in one group across files. Holding out these targets tests frequency transfer; it does not prove that the model memorizes nothing. Nearby frequencies, harmonics and amplitude differences can still affect the comparison.

The next small, explicitly synthetic catalog lets you see the split without downloading data. Its checksums are placeholders and it cannot pass real launch verification.

In [ ]:
toy = FrequencyCatalog(
    source_sha256={f"abra_{family}_{i:04d}.h5": "0" * 64
                   for family in ("training", "validation") for i in range(4)},
    rows=tuple(FrequencyRow(family=family, file=i, segment=s,
                           frequency_hz=1100 + (s % 20) * 100)
               for family in ("training", "validation") for i in range(4) for s in range(200)))
example = make_split(toy, validation_fraction=.2, test_fraction=.2, seed=42)
for name in ("train", "validation", "test"):
    print(name, "frequencies:", getattr(example, name + "_hz"))
    print("  eligible PSD segments per file:", {i: len(s) for i, s in example.population(name).items()})
print("Overlap:", set(example.train_hz) & set(example.validation_hz) | set(example.train_hz) & set(example.test_hz))

### Now apply Trial/Formal proportions to the eligible populations

This **new** task uses ordinary scope selection instead of the frozen paper pool. With 120 eligible training segments per file, Trial `.25` selects 30, and Formal `.5` selects 60. With 40 eligible validation segments, Trial `.5` selects 20 and Formal `1.0` selects 40. The final-test set is not reduced by any of those four settings.

The custom adapter uses `ceil(fraction × eligible count)` per file, at least one segment. Both Trial and Formal select only within their permitted populations. Their scopes need not be nested. `formal_train_portion` stays `1.0`: no second per-epoch reduction.

The 60/20/20 split above counts **distinct frequencies**, not time samples. Repeated frequencies and amplitudes can produce unequal sample counts. Inspect the real counts before running.

In [ ]:
for label, split_name, fraction in [("Trial train", "train", .25), ("Formal train", "train", .5),
                                     ("Trial validation", "validation", .5), ("Formal validation", "validation", 1.)]:
    selected = sample_population(example.population(split_name), fraction, seed=42)
    print(label, {i: len(rows) for i, rows in selected.items()})
print("Final test remains:", {i: len(rows) for i, rows in example.population("test").items()})

## 4. Prepare real data in your project, then inspect its frequencies

Use the official [TIDMAD downloader](https://github.com/jessicafry/TIDMAD), outside both source checkouts:

```bash
python /absolute/path/to/TIDMAD/download_data.py --output_dir /YOUR_PROJECT/data/band-0-3 --train_files 4 --validation_files 4 --science_files 0
```

Keep only `abra_training_0000.h5` through `0003.h5`, the four corresponding validation files, and the initialized `segment_anchors.json` in that directory. These are large data files: one band still represents about 32 GB of uncompressed channel samples. The launch verifies the existing frozen SHA256 manifest. Do not rename/repack/reorder the files: scoring references use original segment indices.

In a terminal in your exp checkout:

```bash
.venv/bin/python -B -m tutorials.paper.tidmad.catalog --data /YOUR_PROJECT/data/band-0-3 --output /YOUR_PROJECT/data/frequency-catalog.json
```

This scans clean channel 2 in all 1,600 original segments. It measures the dominant 1 Hz frequency bin, checks consistency in ten subwindows, and maps the result to the nearest declared nominal tone within 10 Hz. Each row keeps both the nominal label and the observed peak. Ambiguous, changing or unmatched tones are refused. It does **not** infer frequency from a file number or the standalone frequency-reference list. Inspect the proposed groups against the source signal before marking them reviewed. That inspection aid has not yet been qualified on a full real band; a refusal needs investigation, not a relaxed threshold.

Catalog preparation may inspect held-out labels to define the split, but must not use candidate model outputs. Define the split before searching.

In [ ]:
catalog_path = PROJECT / "data/frequency-catalog.json"
real_split = None
if catalog_path.is_file():
    catalog = FrequencyCatalog.model_validate_json(catalog_path.read_text())
    # Change these three values to propose a different scientific split.
    real_split = make_split(catalog, validation_fraction=.2, test_fraction=.2, seed=42)
    for name in ("train", "validation", "test"):
        print(name, {i: len(rows) for i, rows in real_split.population(name).items()})
else:
    print("No real catalog yet. The earlier numbers were a synthetic illustration only.")

## 5. Save your own experiments and scripts

**Task package**: `tasks/tidmad/` owns waveform meaning, frequency eligibility, scoring and validity. **Experiment**: `experiments/*.json` chooses the task, workflow, routing, budgets and output locations.

Set `WRITE_DEMOS=True` to save the three paper-pool examples. Set `WRITE_FREQUENCY_DEMO=True` only after generating and inspecting a real catalog. This adds `tasks/tidmad/compositions/frequency_holdout.yaml`, with the complete split included in the composition identity, plus its experiment and scripts. Re-running writes refuses existing files; choose new names for another study. None of this edits either repository.

`llm/agents.json` is your routing configuration. `advice/README.txt` explains why advice is inactive here: this NoPrior entrypoint disables human advice and data analysis, but **retains literature review**. A text file placed in `advice/` is not automatically used.

In [ ]:
WRITE_DEMOS = False
WRITE_FREQUENCY_DEMO = False
CATALOG_REVIEWED = False

if WRITE_DEMOS:
    for name, settings in [("longer", longer), ("less-trial", less_trial), ("budgets", budgets)]:
        path = PROJECT / "experiments" / f"tidmad-{name}.json"
        with path.open("x") as f:
            f.write(settings.model_dump_json(indent=2))
        write_launcher(PROJECT / "scripts" / f"run-tidmad-{name}.sh", path, base.infra_checkout)

if WRITE_FREQUENCY_DEMO:
    assert real_split is not None and CATALOG_REVIEWED, "Prepare and review the real catalog first"
    composition = write_frequency_task(PROJECT / "tasks/tidmad", real_split)
    holdout = variant(base, "tidmad_frequency_001", protocol="frequency-holdout",
                      composition=str(composition), catalog_reviewed=True,
                      trial_train_fraction=.25, trial_val_fraction=.5,
                      formal_train_fraction=.5, formal_val_fraction=1.)
    path = PROJECT / "experiments/tidmad-frequency.json"
    with path.open("x") as f:
        f.write(holdout.model_dump_json(indent=2))
    write_launcher(PROJECT / "scripts/run-tidmad-frequency.sh", path, base.infra_checkout)
    write_launcher(PROJECT / "scripts/test-tidmad-frequency.sh", path, base.infra_checkout, final=True)
print("Saved files:", [p.name for p in (PROJECT / "experiments").glob("*.json")])

## 6. Check credentials and launch from a terminal

Keep keys in exported environment variables or a trusted external secret store. If using an environment file, keep it outside both repositories and the tutorial project, restrict permissions (`chmod 600`), and source it only if you trust its contents. Do not paste values into notebooks, JSON, shell scripts, outputs or Git.

The next cell displays **names and presence only**. A file existing on disk is not enough: the launching process must inherit the exported variables. Presence does not prove provider access. Jupyter and a separate terminal may have different environments.

In [ ]:
status = credential_status(base)
for name, present in status.items():
    print(name, "present" if present else "MISSING — export in the launching terminal")
if not all(status.values()):
    print("WARNING: launch will refuse missing keys. Do not put secret values in this notebook.")

In the terminal that will launch:

```bash
set -a
source /absolute/path/to/trusted-external-credentials.env
set +a
/YOUR_PROJECT/scripts/run-tidmad.sh               # offline command/config preview
/YOUR_PROJECT/scripts/run-tidmad.sh --launch      # actual search, may spend API credit
# Or, after saving the frequency experiment:
/YOUR_PROJECT/scripts/run-tidmad-frequency.sh --launch
```

The scripts accept one RTX 5090 or H100 and check CUDA plus VRAM headroom. Choose `gpu` and budgets in your experiment. A different NVIDIA model requires qualifying and extending the launcher GPU allowlist, checking the pinned CUDA environment and doing a small calibration run; changing a name alone is not qualification. AMD and Intel GPUs are unsupported.

This fixed workflow has no separate security sandbox. Generated model code runs with your local permissions. `runs/<run_name>/` receives search artifacts, generated models and results; the adjacent `.tutorial.json` records settings and checks. Time budgets are not total wall-time/API-cost caps. Results remain diagnostic and existing blocking Health rules stay active.

## 7. Final test: select once, stop search, then evaluate

Workflow validation is visible to the agent and can influence models, hyperparameters and epoch selection. It is **not** the final test. Use validation plus the run's Health evidence to select one completed model, stop the search, then create `final-test/candidate.json` in your project:

```json
{
  "model_name": "YOUR_SELECTED_MODEL_REGISTRY_NAME",
  "exp_id": "YOUR_SELECTED_ATTEMPT_ID",
  "checkpoint": "/YOUR_PROJECT/runs/tidmad_frequency_001/PATH_TO_SELECTED.pth",
  "model_config_path": "/YOUR_PROJECT/runs/tidmad_frequency_001/PATH_TO_MODEL_CONFIG.json",
  "loss_config_path": "/YOUR_PROJECT/runs/tidmad_frequency_001/PATH_TO_LOSS_CONFIG.json",
  "generated_library": "/YOUR_PROJECT/runs/tidmad_frequency_001/generated_library",
  "search_completed": true,
  "inference_batch_size": 1
}
```

Use the **actual selected attempt's** paths and registry name from its artifacts/native inference command; do not choose files by modification time or copy these placeholders literally. Keep its native `_OK_<exp_id>` training-success marker next to the checkpoint. The final runner reuses native model loading and task scoring; it does not retrain or call an LLM.

```bash
/YOUR_PROJECT/scripts/test-tidmad-frequency.sh --candidate /YOUR_PROJECT/final-test/candidate.json --output /YOUR_PROJECT/final-test/test-001 --seal
/YOUR_PROJECT/scripts/test-tidmad-frequency.sh --candidate /YOUR_PROJECT/final-test/candidate.json --output /YOUR_PROJECT/final-test/test-001 --evaluate
```

`--seal` records model/config/plugin/task hashes before test inference. `--evaluate` requires the same selection, uses every final-test segment, and writes `result.json` separately from the search. A started evaluation cannot be silently rerun in that output directory. A failed attempt leaves its evidence for inspection.

This is a **procedural holdout**, not a private evaluator: source HDF5 files and split labels are locally readable. The script cannot prove that you stopped another process or never inspected test results elsewhere. Do not use the test score to pick another model, alter the split, or continue searching. If you do, it becomes development data and you need a new final test.

The standalone final report is diagnostic: it computes the existing TIDMAD metric but does not run the full workflow Health assessment. Do not present it as a paper-qualified or Health-approved score.

## What has and has not been demonstrated

Configuration and frequency-scope tests establish that workflow scopes exclude final-test frequencies and retain original segment identities. Notebook execution can be checked offline. They do not establish real-data catalog accuracy, scientific performance, or GPU feasibility for your chosen budgets. A full real TIDMAD band and an API/GPU search are still needed to qualify those claims.